# Act 1 — NoSQL: Firestore and Bigtable

When relational doesn't fit, GCP offers two very different NoSQL options: **Firestore** for document-shaped data with real-time features, and **Bigtable** for very-wide-column data at extreme scale. They look superficially similar (both NoSQL, both managed) and have almost nothing in common operationally. Pick on data shape and access pattern, not on the word "NoSQL."

## Firestore

**Firestore** is GCP's document database. Data is organised as **collections** of **documents**; documents are nested JSON-like maps; collections can be nested under documents (sub-collections).

**Two modes (irreversibly chosen at database creation):**

- **Native mode** — the modern Firestore. Real-time listeners, mobile and web client SDKs with security rules, strong consistency on document reads. The default for new projects.
- **Datastore mode** — the legacy App Engine Datastore API surface, kept for backward compatibility. Don't pick this for new work.

**Strengths:**

- **Real-time listeners** — clients subscribe to a query and get push updates on writes. The substrate behind a lot of consumer apps (chat, collaborative editing, live dashboards).
- **Offline mode** — mobile/web SDKs cache locally and sync when the connection returns.
- **Security rules** — declarative auth rules that run on the server, scoped per document/path. Lets you ship mobile apps that talk to Firestore directly without a backend.
- **Multi-region replication** — `nam5`, `eur3` configurations replicate across regions for HA.

**Limits to watch:**

- **Composite indexes are explicit.** Queries that filter on multiple fields need an index you declare; the SDK error tells you what to create.
- **Single-document throughput.** A single document is a hot spot — sustained writes >1/sec on the same doc need re-modelling.
- **No JOIN.** You denormalise.

**When Firestore is right:** mobile/web app backends, real-time collaborative features, semi-structured documents (orders, profiles, content). When it's wrong: heavy relational queries, analytical workloads, or sustained high write throughput on individual documents.

## Bigtable

**Bigtable** is GCP's wide-column NoSQL database, the same engine Google uses internally for Search, Maps, and Gmail. Wire-compatible with the HBase API. It scales horizontally to petabytes and millions of reads/writes per second on a single cluster.

**The data model:**

- **Tables** contain rows; rows are identified by a single **row key** (a byte string).
- Each row has one or more **column families**; each family contains arbitrary columns.
- Cells (row × column) can hold multiple versions over time.

**The single most important design decision: row key design.** Bigtable distributes rows across nodes by row key prefix. A row key that puts all of yesterday's writes in the same prefix creates a hot tablet; a row key that mixes recency with another high-cardinality field spreads load. Real-world keys often look like `user_id#reverse_timestamp` to scatter writes while keeping per-user reads contiguous.

**Operational shape:**

- **Cluster nodes** — you pick how many; you can scale up or down. Autoscaling is available based on CPU.
- **Replication** — multi-cluster instances replicate across zones or regions. Reads can route to the nearest replica; writes propagate asynchronously.
- **App profiles** — per-application routing policies (single-cluster or multi-cluster, failover behaviour).

**When Bigtable is right:** time-series data, IoT telemetry, ad-tech, fraud detection — anything with very high write throughput, key-based access, and TB-to-PB scale. **When it's wrong:** small datasets (Bigtable charges by node-hour; minimum cost is meaningful), or anything needing rich queries, secondary indexes, or transactions across rows.

# Act 2 — Spanner: globally distributed strong consistency

Spanner is the GCP database that has no equivalent on AWS or Azure. It's a SQL database that scales horizontally across regions with **strong consistency** and **ACID transactions** — including transactions that span multiple regions. The trick is **TrueTime**, a Google-internal time API backed by GPS and atomic clocks that gives the storage layer a globally-consistent notion of "now."

This is GCP's most differentiated database product. It's also the most expensive at small scale. Understanding when to reach for it (and when not to) is the headline of this act.

## Spanner — what it is and how it scales

**Architecturally:** Spanner separates compute (query processing) from storage (replicated, distributed). You provision **nodes** (or processing units) — the engine scales horizontally as you add them. Storage is automatically sharded into **splits** by primary key.

**Three region configurations:**

- **Regional** — three replicas in three zones of one region. Cheaper, comparable to Cloud SQL HA in shape.
- **Dual-region** — two regions with five replicas. RTO/RPO improvements over single-region.
- **Multi-region** — three or more regions; the headline mode. Synchronous writes globally, with read-only replicas in additional regions. Sub-second writes across continents (Paxos-coordinated through TrueTime).

**Schema features that matter:**

- **Interleaved tables** — physically co-locate child rows with their parent (`Customer.orders` interleaved under `Customer`). Joins across the interleave are local to one split.
- **Secondary indexes** — global by default; can be made interleaved (local to a split) for performance.
- **Foreign keys, check constraints, schema migrations online** — Spanner is a full relational database, not just a key-value store with SQL skin.

**Spanner vs Cloud SQL — when each is right:**

- Cloud SQL: single-region OLTP, tens of TBs, ordinary scale. Pay-as-you-go.
- Spanner: horizontal write scale, multi-region writes, very large datasets, very high transactional throughput. Always expensive.

**Cost shape.** Spanner bills per node-hour (or per processing unit for very small instances). A small regional Spanner is ~$700/month minimum — significantly more than Cloud SQL at the same scale. The crossover where Spanner becomes the obvious choice is around the point where Cloud SQL needs read replicas, write sharding, or multi-region active-active.

# Act 3 — BigQuery: the analytics warehouse

BigQuery is GCP's data warehouse. It's the product most teams who use GCP at all eventually adopt — even teams whose application database is on AWS RDS or Azure SQL often end up pulling data into BigQuery for analytics. The shape is sufficiently different from a traditional warehouse (Redshift, Synapse, Snowflake) that it's worth understanding from first principles.

## BigQuery architecture — storage and compute are separate

**The headline architectural fact**: BigQuery's storage layer (Capacitor, columnar) and its query engine (Dremel) are entirely separate. You pay for them independently. Adding storage doesn't add compute capacity; adding compute capacity (slots) doesn't grow storage.

This matters because it changes the cost shape:

- **Storage cost** — per-GB-month for active tables; lower per-GB-month for tables not modified in 90 days (automatic).
- **Compute cost** — two pricing models, picked per query or per organisation:
  - **On-demand** — pay per TB scanned by the query. Free to start, predictable per-query, but a runaway query is expensive.
  - **Editions (Capacity)** — pre-purchase slot-hours via Standard / Enterprise / Enterprise Plus tiers, with autoscaling slots. Cheaper at scale, requires commitment.

**Slots** are the unit of compute. One slot is roughly one virtual CPU's worth of query execution. Editions reserve a baseline of slots; autoscaling adds more on demand within a configured ceiling. Slots are dynamic at the query level — a single big query can briefly use the entire reservation.

## Partitioning, clustering, materialised views

The two table-design knobs that matter for cost and performance:

- **Partitioning** — split a table by date (`ingestion_time` or a `DATE` column) or by integer range. Queries that filter on the partition column scan only the relevant partitions. **Pruning** is the term for this; aggressive partitioning + pruning is the single biggest cost-control lever in BigQuery.
- **Clustering** — sort data within each partition by up to four columns. Queries that filter on clustered columns can skip data blocks without scanning them. Use for high-cardinality columns commonly used in `WHERE` clauses (user_id, customer_id, event_type).

**Materialised views** maintain a precomputed query result that updates as the base table changes. The query optimiser automatically rewrites queries to hit the MV if it matches. Use for aggregations that hit many queries (daily revenue per region, top-N).

**BI Engine** is an in-memory accelerator for BI workloads — reserve a memory amount; matching queries serve from the in-memory cache sub-second. Mostly transparent: BI tools (Looker, Looker Studio, Tableau) benefit automatically.

## Other BigQuery features worth naming

- **Federated queries** — query data in Cloud SQL, Spanner, GCS, Bigtable directly from BigQuery without ETL. Useful for ad-hoc exploration, not for steady-state pipelines.
- **Streaming inserts** — append rows in real time via the Storage Write API. Latencies on the order of seconds; supports Pub/Sub-to-BigQuery direct subscriptions.
- **BigQuery ML (BQML)** — train and inference linear/logistic regression, k-means, time-series forecasting, and call Vertex AI models from SQL. Lower bar to entry for analytics teams than Vertex AI proper.
- **BigQuery Omni** — query data in S3 (AWS) or Azure Blob from BigQuery, where the compute runs in the other cloud. Used for multi-cloud analytics without copying data.
- **Authorized views** — share a view (defined as a SQL query over base tables) without granting access to the base tables. Standard fine-grained-access pattern.

# Act 4 — Pipelines and BI

Data rarely arrives in its final shape. Three GCP products handle the pipelines between source systems and analytical stores: **Dataflow** (streaming/batch unified), **Dataproc** (managed Spark/Hadoop), and **Pub/Sub** (which we'll meet properly in notebook 10). Looker and Looker Studio close out with a one-liner on BI.

## Dataflow — Apache Beam, batch and streaming unified

**Dataflow** runs Apache Beam pipelines as fully-managed jobs. Beam's value is that the same pipeline code runs batch *and* streaming — windowing, watermarks, late-data handling are core API concepts.

**Key concepts:**

- **PCollection** — a dataset, bounded (batch) or unbounded (streaming).
- **PTransform** — an operation on PCollections (Map, GroupByKey, Combine, Window, …).
- **Window** — a slice of an unbounded PCollection (fixed, sliding, session windows).
- **Watermark** — Beam's estimate of "event time we've seen up to." Drives when windows close.

**Streaming Engine** is Dataflow's modern execution backend: state separated from compute, autoscaling smoother, snapshotting supported.

**Common shape:** Pub/Sub → Dataflow → BigQuery. A streaming pipeline pulls messages from Pub/Sub, transforms them (parse, enrich, deduplicate), writes to BigQuery streaming inserts. Beam's windowed aggregations let you emit per-minute or per-hour rollups.

## Dataproc — managed Spark and Hadoop

**Dataproc** runs Apache Spark, Hadoop, Hive, Presto, and friends on GCP-managed clusters. Used by teams with existing Spark/Hadoop code that they want to migrate to the cloud without rewriting.

Three flavours:

- **Dataproc clusters** — long-running or ephemeral clusters you provision and manage.
- **Dataproc Serverless** — submit a Spark job, GCP runs it without you provisioning a cluster. The right default for new work.
- **Dataproc Metastore** — managed Hive metastore as a service, for sharing schemas across multiple compute engines.

**Choose-what:** Dataflow for new pipelines, Beam-shaped code. Dataproc for existing Spark / Hadoop investments that would be costly to rewrite. New analytical work in 2026 is usually better served by Dataflow + BigQuery.

## Looker and Looker Studio

**Looker** is Google's full BI platform with a semantic layer (LookML) — used by data teams that want governed metrics, embedded analytics, and rich dashboarding. Paid product.

**Looker Studio** (formerly Data Studio) is the free, lightweight reporting tool — drag-and-drop dashboards over BigQuery, GCS, GA4, and other sources. Used widely for internal dashboards where the rigor of Looker isn't needed.

## Choose-what — NoSQL & Analytics at a glance

1. **Document-shaped data with real-time/offline needs?** → Firestore.
2. **Time-series, IoT, wide-column at extreme scale?** → Bigtable.
3. **Globally distributed SQL with ACID transactions?** → Spanner.
4. **Analytical SQL over TBs+ of data?** → BigQuery.
5. **ETL / streaming pipelines?** → Dataflow (Beam) for new, Dataproc for existing Spark.
6. **Self-serve dashboards?** → Looker Studio for ad-hoc, Looker for governed.

Notebook 10 picks up Pub/Sub and the integration / messaging surface in detail.

## What carries into later chapters

BigQuery is the most-touched analytical store in the rest of this course. Cloud Audit Logs export to BigQuery for analysis (notebook 12). Billing data exports to BigQuery via the BQ billing export (notebook 14). Cloud Logging routes log buckets to BigQuery sinks for SQL-over-logs.

Three habits to carry forward:

- **Partition BigQuery tables on a date column.** Untouched tables become expensive faster than expected; partitioned ones scale gracefully.
- **Spanner only when global writes are real.** It's the right answer to a specific problem and overkill for most.
- **Firestore documents are not relational rows.** Denormalise; design for read patterns; expect to model differently than you would in Postgres.